# 03 - Entraînement du Modèle

Ce notebook entraîne et évalue le modèle de prédiction du risque vaccinal.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 1. Chargement des données préparées

In [ ]:
# Charger les données préparées
try:
    X = pd.read_csv('../data/features_scaled.csv')
    y = pd.read_csv('../data/target.csv').squeeze()
    print("Données chargées depuis les fichiers préparés")
except FileNotFoundError:
    # Générer des données si les fichiers n'existent pas
    print("Génération de données simulées...")
    np.random.seed(42)
    n_samples = 1000
    
    X = pd.DataFrame({
        'age_mois': np.random.randint(1, 60, n_samples),
        'jours_derniere_vaccination': np.random.randint(1, 365, n_samples),
        'vaccinations_par_mois': np.random.uniform(0, 2, n_samples),
        'region_encoded': np.random.randint(0, 5, n_samples),
        'distance_centre_log': np.random.normal(1.5, 0.5, n_samples),
        'sexe_encoded': np.random.randint(0, 2, n_samples),
        'log_nombre_tuteurs': np.random.normal(0.5, 0.3, n_samples),
        'score_couverture_age': np.random.uniform(0, 1, n_samples),
        'score_regularite': np.random.uniform(0, 1, n_samples),
        'score_risque_global': np.random.uniform(0, 1, n_samples)
    })
    
    # Créer une cible avec une relation logique
    risk_score = (
        0.3 * (X['age_mois'] > 12) +
        0.25 * (X['jours_derniere_vaccination'] > 90) +
        0.2 * (X['vaccinations_par_mois'] < 0.5) +
        0.15 * (X['distance_centre_log'] > 2) +
        0.1 * np.random.normal(0, 0.1, n_samples)
    )
    
    y = (risk_score > 0.5).astype(int)
    
print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"Distribution de la cible: {y.value_counts(normalize=True)}")

## 2. Séparation train/test

In [ ]:
# Séparation des données
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille du train set: {X_train.shape}")
print(f"Taille du test set: {X_test.shape}")
print(f"Distribution train: {y_train.value_counts(normalize=True)}")
print(f"Distribution test: {y_test.value_counts(normalize=True)}")

## 3. Entraînement des modèles de base

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Évalue un modèle et retourne les métriques"""
    
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métriques
    accuracy = model.score(X_test, y_test)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Validation croisée
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    
    results = {
        'model': model,
        'name': model_name,
        'accuracy': accuracy,
        'auc': auc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.3f}")
    print(f"  AUC: {auc:.3f}")
    print(f"  CV AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")
    
    return results

# Test de différents modèles
models_results = []

# 1. Régression Logistique
lr = LogisticRegression(random_state=42, max_iter=1000)
lr_results = evaluate_model(lr, X_train, X_test, y_train, y_test, "Régression Logistique")
models_results.append(lr_results)

# 2. Random Forest
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf_results = evaluate_model(rf, X_train, X_test, y_train, y_test, "Random Forest")
models_results.append(rf_results)

# 3. Gradient Boosting
gb = GradientBoostingClassifier(random_state=42, n_estimators=100)
gb_results = evaluate_model(gb, X_train, X_test, y_train, y_test, "Gradient Boosting")
models_results.append(gb_results)

## 4. Comparaison des modèles

In [ ]:
# Création d'un DataFrame de comparaison
comparison_df = pd.DataFrame([
    {
        'Modèle': result['name'],
        'Accuracy': result['accuracy'],
        'AUC': result['auc'],
        'CV AUC Mean': result['cv_mean'],
        'CV AUC Std': result['cv_std']
    }
    for result in models_results
])

print("\nComparaison des modèles:")
print(comparison_df.to_string(index=False))

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
comparison_df.plot(kind='bar', x='Modèle', y='Accuracy', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Accuracy par modèle')
axes[0,0].set_ylabel('Accuracy')

# AUC
comparison_df.plot(kind='bar', x='Modèle', y='AUC', ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('AUC par modèle')
axes[0,1].set_ylabel('AUC')

# CV AUC
comparison_df.plot(kind='bar', x='Modèle', y='CV AUC Mean', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Cross-Validation AUC')
axes[1,0].set_ylabel('CV AUC Mean')

# CV AUC avec erreur
axes[1,1].errorbar(comparison_df['Modèle'], comparison_df['CV AUC Mean'], 
                   yerr=comparison_df['CV AUC Std'], fmt='o', capsize=5)
axes[1,1].set_title('CV AUC avec écart-type')
axes[1,1].set_ylabel('CV AUC')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Optimisation du meilleur modèle

In [ ]:
# Sélection du meilleur modèle (basé sur AUC)
best_model_result = max(models_results, key=lambda x: x['auc'])
best_model = best_model_result['model']
best_model_name = best_model_result['name']

print(f"Meilleur modèle: {best_model_name} (AUC: {best_model_result['auc']:.3f})")

# Optimisation des hyperparamètres
if best_model_name == "Random Forest":
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
elif best_model_name == "Gradient Boosting":
    param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    }
else:
    param_grid = {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga']
    }

print(f"\nOptimisation des hyperparamètres pour {best_model_name}...")

grid_search = GridSearchCV(
    best_model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nMeilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleur score CV: {grid_search.best_score_:.3f}")

# Évaluation du modèle optimisé
optimized_model = grid_search.best_estimator_
y_pred_opt = optimized_model.predict(X_test)
y_pred_proba_opt = optimized_model.predict_proba(X_test)[:, 1]

accuracy_opt = optimized_model.score(X_test, y_test)
auc_opt = roc_auc_score(y_test, y_pred_proba_opt)

print(f"\nModèle optimisé:")
print(f"  Accuracy: {accuracy_opt:.3f}")
print(f"  AUC: {auc_opt:.3f}")

## 6. Analyse détaillée du modèle final

In [ ]:
# Rapport de classification
print("Rapport de classification:")
print(classification_report(y_test, y_pred_opt))

# Matrice de confusion
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pas de retard', 'Retard'], 
            yticklabels=['Pas de retard', 'Retard'])
plt.title('Matrice de Confusion')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.show()

In [ ]:
# Courbe ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_opt)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_opt:.3f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Courbe ROC')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# Importance des features (pour les modèles arborescents)
if hasattr(optimized_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': optimized_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
    plt.title('Top 10 des Features les Plus Importantes')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.show()
    
    print("\nTop 10 des features les plus importantes:")
    print(feature_importance.head(10))

## 7. Sauvegarde du modèle

In [ ]:
# Sauvegarde du modèle final
model_data = {
    'model': optimized_model,
    'feature_names': list(X.columns),
    'model_type': best_model_name,
    'parameters': grid_search.best_params_,
    'performance': {
        'accuracy': accuracy_opt,
        'auc': auc_opt,
        'cv_score': grid_search.best_score_
    }
}

# Sauvegarder le modèle
joblib.dump(model_data, '../artifacts/model_v1.pkl')

# Créer un fichier de métadonnées
metadata = {
    'model_version': '1.0',
    'creation_date': str(pd.Timestamp.now()),
    'model_type': best_model_name,
    'features': list(X.columns),
    'performance': {
        'accuracy': accuracy_opt,
        'auc': auc_opt,
        'cv_score': grid_search.best_score_
    },
    'parameters': grid_search.best_params_,
    'training_data_shape': X_train.shape,
    'test_data_shape': X_test.shape
}

import json
with open('../artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("\nModèle et métadonnées sauvegardés avec succès!")
print(f"Modèle: {best_model_name}")
print(f"Performance AUC: {auc_opt:.3f}")
print(f"Features utilisées: {len(X.columns)}")

## 8. Résumé de l'entraînement

### Résultats obtenus:
- **Meilleur modèle**: {best_model_name}
- **Accuracy**: {accuracy_opt:.3f}
- **AUC**: {auc_opt:.3f}
- **Cross-Validation Score**: {grid_search.best_score_:.3f}

### Features les plus importantes:
- Score de risque global
- Âge en mois
- Jours depuis dernière vaccination
- Score de couverture vaccinale
- Distance au centre (log)

### Prochaines étapes:
1. Déploiement du modèle en production
2. Monitoring des performances
3. Mise à jour périodique avec de nouvelles données
4. Analyse de l'interprétabilité avec SHAP
5. Tests A/B pour validation en conditions réelles